In [2]:
import pandas as pd

df = pd.read_csv("CRMLSListing_merge.csv")
df.head(5)

/var/folders/d_/gll3jlg1445_2nm3xgq_xvdc0000gn/T/ipykernel_7163/3028586889.py:3: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("CRMLSListing_merge.csv")


,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,Latitude,Longitude,UnparsedAddress,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName.1,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,UnparsedAddress.1,month
0,90000.0,1075010398,miriamlara03@gmail.com,NaN,NaN,Miriam,Lara,34.097939,-117.909653,1045 N Azusa 61,...,NaN,2.0,Covina Valley Unified,91722,NaN,0.0,NaN,NaN,1045 N Azusa 61,202401
1,1500000.0,1074974457,janelle@judsonre.com,NaN,NaN,Janelle,Judson,33.121241,-117.081614,NaN,...,NaN,NaN,NaN,92025,NaN,0.0,NaN,NaN,NaN,202401
2,1340000.0,1074973329,haleh360@Gmail.com,NaN,NaN,Haleh,Dowlatshahi,34.052207,-118.408445,2220 Avenue Of The Stars 2704,...,False,NaN,NaN,90067,NaN,2105.0,177861.0,NaN,2220 Avenue Of The Stars 2704,202401
3,2500000.0,1074954552,Reneechen@yourhomesoldguaranteed.com,NaN,NaN,Renee,Chen,33.496363,-117.691677,16 Palisades,...,False,3.0,Capistrano Unified,92677,NaN,254.0,5300.0,NaN,16 Palisades,202401
4,3150000.0,1074936537,anader@dppre.com,NaN,NaN,Margaret,Nader,34.119345,-118.111254,1615 Waverly Road,...,NaN,2.0,NaN,91108,NaN,NaN,9404.0,NaN,1615 Waverly Road,202401


In [5]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns
print("Categorical columns:", len(cat_cols))
print(cat_cols)


Categorical columns: 48
Index(['ListAgentEmail', 'CloseDate', 'ListAgentFirstName',
       'ListAgentLastName', 'UnparsedAddress', 'PropertyType',
       'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'AssociationFeeFrequency', 'MLSAreaMajor', 'CountyOrParish',
       'PropertyType.1', 'MlsStatus', 'ElementarySchool',
       'ListAgentFirstName.1', 'AttachedGarageYN', 'BuilderName',
       'PropertySubType', 'SubdivisionName', 'BuyerOfficeAOR',
       'BuyerAgencyCompensationType', 'ListingId', 'City',
       'ContractStatusChangeDate', 'CoBuyerAgentFirstName',
       'PurchaseContractDate', 'ListingContractDate', 'BusinessType',
       'StateOrProvince', 'MiddleOrJuniorSchool', 'FireplaceYN', 'HighSchool',
       'Levels', 'ListAgentLastName.1', 'CloseDate.1', 'LotSizeDimensions',
       'NewConstructionYN', 'HighSchoolDistr

In [9]:
import pandas as pd

# =========================
# 1. Basic Info
# =========================
print("=== DATA SHAPE ===")
print(df.shape)

# 2. Build categorical review table
# =========================
cat_review = pd.DataFrame({
    "column": cat_cols,
    "missing_count": [df[col].isna().sum() for col in cat_cols],
    "missing_pct": [round(df[col].isna().mean() * 100, 2) for col in cat_cols],
    "n_unique": [df[col].nunique(dropna=True) for col in cat_cols]
})

# =========================
# 3. Sample top values
# =========================
top_values = []
for col in cat_cols:
    vals = df[col].value_counts(dropna=False).head(5)
    vals_str = "; ".join([f"{str(idx)} ({cnt})" for idx, cnt in vals.items()])
    top_values.append(vals_str)

cat_review["top_values_sample"] = top_values

# =========================
# 4. Core columns
# =========================
core_columns = [
    "PropertyType", "PropertySubType", "City", "PostalCode",
    "CountyOrParish", "StateOrProvince", "MLSAreaMajor",
    "MlsStatus", "ListingId", "UnparsedAddress",
    "ListingContractDate", "ContractStatusChangeDate",
    "PurchaseContractDate", "OriginatingSystemName",
    "OriginatingSystemSubName"
]

# =========================
# 5. Decision
# =========================
def classify_column(row):
    col = row["column"]
    missing_pct = row["missing_pct"]
    n_unique = row["n_unique"]

    if col in core_columns:
        return pd.Series(["Keep (Core)"])
    elif missing_pct >= 90:
        return pd.Series(["Drop"])
    elif n_unique == 1:
        return pd.Series(["Drop"])
    elif "Name" in col or "Email" in col:
        return pd.Series(["Drop"])
    else:
        return pd.Series(["Review"])

cat_review[["decision"]] = cat_review.apply(classify_column, axis=1)

# =========================
# 6. Duplicate Check (overall)
# =========================
total_duplicates = df.duplicated().sum()
dup_pct = round(total_duplicates / len(df) * 100, 2)

print("\n=== DUPLICATE ROWS ===")
print(f"Total duplicate rows: {total_duplicates}")
print(f"Duplicate %: {dup_pct}%")

# =========================
# 7. Duplicate Check (key columns)
# =========================
print("\n=== DUPLICATE CHECK BY KEY COLUMNS ===")

if "ListingId" in df.columns:
    print("Duplicate ListingId:", df["ListingId"].duplicated().sum())

if "UnparsedAddress" in df.columns:
    print("Duplicate Address:", df["UnparsedAddress"].duplicated().sum())

if "latfilled" in df.columns and "lonfilled" in df.columns:
    print("Duplicate Geo:", df.duplicated(subset=["latfilled", "lonfilled"]).sum())

# =========================
# 8. Sort table
# =========================
cat_review = cat_review.sort_values(by="missing_pct", ascending=False).reset_index(drop=True)

# =========================
# 9. Display
# =========================
print("\n=== KEEP (CORE) ===")
display(cat_review[cat_review["decision"] == "Keep (Core)"])

print("\n=== DROP ===")
display(cat_review[cat_review["decision"] == "Drop"])

print("\n=== REVIEW ===")
display(cat_review[cat_review["decision"] == "Review"])

=== DATA SHAPE ===
(853863, 85)

=== DUPLICATE ROWS ===
Total duplicate rows: 0
Duplicate %: 0.0%

=== DUPLICATE CHECK BY KEY COLUMNS ===
Duplicate ListingId: 151
Duplicate Address: 140791

=== KEEP (CORE) ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
21,PurchaseContractDate,486031,56.92,831,nan (486031); 2024-05-01 (1227); 2024-04-24 (1...,Keep (Core)
27,PropertySubType,104169,12.20,34,SingleFamilyResidence (478795); Condominium (1...,Keep (Core)
28,MLSAreaMajor,96427,11.29,1153,nan (96427); 699 - Not Defined (85016); SRCAR ...,Keep (Core)
30,ContractStatusChangeDate,6932,0.81,848,nan (6932); 2024-04-30 (3242); 2024-05-01 (316...,Keep (Core)
33,UnparsedAddress,2159,0.25,713071,nan (2159); 0 0 (147); 1225 Vienna Drive (73);...,Keep (Core)
35,City,962,0.11,1324,Los Angeles (68718); San Diego (33859); San Jo...,Keep (Core)
37,PostalCode,207,0.02,5298,92253 (5310); 92618 (4557); 92262 (4363); 9221...,Keep (Core)
38,StateOrProvince,66,0.01,41,CA (853500); OS (117); nan (66); AZ (26); NV (16),Keep (Core)
41,MlsStatus,0,0.00,5,Active (471530); Closed (232327); Pending (101...,Keep (Core)
43,CountyOrParish,1,0.00,63,Los Angeles (266193); Riverside (110304); Oran...,Keep (Core)



=== DROP ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
0,BusinessType,847887,99.30,1910,nan (847887); Restaurant (839); Retail (295); ...,Drop
1,CoBuyerAgentFirstName,834223,97.70,3188,nan (834223); Michael (259); David (211); Jenn...,Drop
2,BuilderName,824561,96.57,4602,nan (824561); Lennar (2570); Toll Brothers (13...,Drop
3,LotSizeDimensions,805156,94.30,17395,nan (805156); 50x135 (896); 50x150 (689); 50x1...,Drop
4,ElementarySchool,774565,90.71,1446,nan (774565); Other (14999); Harbor View (690)...,Drop
5,MiddleOrJuniorSchool,774474,90.70,679,nan (774474); Other (11277); Corona Del Mar (1...,Drop
8,CoListAgentFirstName,683557,80.05,8411,nan (683557); Michael (2696); David (2362); Jo...,Drop
9,CoListAgentLastName,683070,80.00,17338,nan (683070); Myatt (1134); Smith (1020); Lee ...,Drop
10,CoListOfficeName,682926,79.98,7084,nan (682926); Compass (19357); Coldwell Banker...,Drop
12,BuyerOfficeName,609049,71.33,16936,nan (609049); Compass (17104); Coldwell Banker...,Drop



=== REVIEW ===


,column,missing_count,missing_pct,n_unique,top_values_sample,decision
6,HighSchool,749518,87.78,534,nan (749518); Other (4692); Portola (1701); Te...,Review
7,BuyerAgencyCompensationType,730325,85.53,3,nan (730325); Item1 (113085); Item (10117); Se...,Review
11,BuyerOfficeAOR,620800,72.70,66,nan (620800); OrangeCounty (22370); SanDiego (...,Review
15,BuyerAgentMlsId,607465,71.14,89235,nan (607465); NONMEMBER (3668); NONE (2912); n...,Review
16,AssociationFeeFrequency,607453,71.14,4,nan (607453); Monthly (230622); Annually (1085...,Review
18,CloseDate,598619,70.11,917,nan (598619); 2024-03-29 (1375); 2025-10-31 (1...,Review
19,CloseDate.1,598619,70.11,917,nan (598619); 2024-03-29 (1375); 2025-10-31 (1...,Review
22,HighSchoolDistrict,339481,39.76,454,nan (339481); Other (55198); Los Angeles Unifi...,Review
23,AttachedGarageYN,291120,34.09,2,True (432628); nan (291120); False (130115),Review
24,Levels,164094,19.22,19,One (395065); Two (230719); nan (164094); Thre...,Review


##Second Round of Screening: Review Phase

Drop:
BuyerAgencyCompensationType，BuyerOfficeAOR, BuyerAgentMlsId, AssociationFeeFrequency, FireplaceYN

Duplicates (Drop): CloseDate.1,UnparsedAddress.1,PropertyType.1

keep:
HighSchool- School District Housing

CloseDate- With for-sale listings, the date the purchase agreement was fulfilled. 

HighSchoolDistrict	- The name of the high school district having a catchment area that includes the a ssociated property. 

AttachedGarageYN- A marker used to indicate whether a garage is attached to the main residence.
Levels- the number of levels in the property being sold. 

NewConstructionYN-- Is the property newly constructed and has not been
previously occupied?






